# ALOS-2 forested-wetland inundation for an ROI

Clips the Rosenqvist et al. (2026) ALOS-2 ScanSAR inundation-duration mosaic
(Zenodo [10.5281/zenodo.22981938](https://doi.org/10.5281/zenodo.22981938), v1.0)
to a bounding box from `rois.json` and maps **inundated vegetation**.

Each pixel is the number of observations (out of 99, 2015–2025) classified as
inundated vegetation. DN 0 is both “never inundated” and no-data, so it is not
counted as flooded. On the Central Africa mosaic, DN 1–3 are filtered and are
not counted either. The mosaic is chosen from the ROI centre, so the same call
works for South America and Central Africa.

GLWD v2 classes 1–21 (open water through ephemeral wetland) are drawn under the inundation at 70% opacity, and only where they occur in the ROI. The legend lists those classes. The rest of the ROI is black. Inundation uses `inferno`: rare detections stay dark and frequent flooding is yellow, so it stays readable on black.

Set `ROI_NAMES` and run the last cell. One figure per ROI.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# All-Years inundation-duration mosaics (50 m, EPSG:4326). Add a file here
# when another region is downloaded; the ROI picks the mosaic that covers it.
ALOS2_DIR = Path("/home/georg/data/Biomass_hackathon/ALOS_2_flooded")
ROI_JSON = Path("../rois.json")
OUT_DIR = Path("../outputs/alos2_flooded")

# GLWD v2 main class (EPSG:4326). Open water drawn under the inundation.
GLWD_TIF = Path(
    "/home/georg/data/Biomass_hackathon/GLWD_v2_0_combined_classes_tif"
    "/GLWD_v2_0_combined_classes/GLWD_v2_0_main_class.tif"
)
GLWD_LEGEND = Path(
    "/home/georg/data/Biomass_hackathon/GLWD_v2_0_combined_classes_tif/GLWD_Legend_v2_0.csv"
)
# GLWD classes 1-21. A class is drawn only if it occurs in the ROI, at this opacity.
WATER_CLASS_IDS = tuple(range(1, 22))
WATER_ALPHA = 0.70
WATER_RGB = {
    1: (30, 90, 190),
    2: (70, 120, 170),
    3: (40, 140, 180),
    4: (45, 140, 235),
    5: (70, 160, 220),
    6: (120, 180, 220),
    7: (150, 210, 230),
    8: (20, 90, 60),
    9: (110, 150, 90),
    10: (0, 110, 70),
    11: (130, 200, 140),
    12: (210, 110, 30),
    13: (240, 190, 70),
    14: (130, 80, 40),
    15: (190, 160, 110),
    16: (0, 80, 50),
    17: (80, 160, 100),
    18: (90, 130, 80),
    19: (170, 190, 130),
    20: (170, 80, 100),
    21: (210, 160, 160),
}

# None = product minimum (1 in South America, 4 in Central Africa).
# A number is "inundated in at least this many of the 99 observations".
MIN_COUNT = None

ROI_NAMES = [
    "Belém",
    "Lac Télé Community Reserve (LTCR)",
]

In [ ]:
import csv
import json
import re
import unicodedata

import rasterio
from matplotlib.patches import Patch
from rasterio.windows import Window, from_bounds


def load_roi(name, roi_json=ROI_JSON):
    """Return (name, (west, south, east, north)) from rois.json."""
    rois = json.loads(Path(roi_json).read_text())
    key = name.casefold().strip()
    hits = [r for r in rois if r["name"].casefold().strip() == key]
    if not hits:
        hits = [r for r in rois if key in r["name"].casefold()]
    if len(hits) != 1:
        known = ", ".join(r["name"] for r in rois)
        raise ValueError(f"ROI {name!r} matched {len(hits)} entries. Known: {known}")
    b = hits[0]["bbox"]
    return hits[0]["name"], (b["west"], b["south"], b["east"], b["north"])


def mosaic_paths(alos2_dir=ALOS2_DIR):
    paths = sorted(Path(alos2_dir).glob("alos2_inund-dur_*_All-Years_*.tif"))
    if not paths:
        raise FileNotFoundError(f"No All-Years inundation mosaics in {alos2_dir}")
    return paths


def pick_mosaic(bbox, paths):
    """Mosaic with the largest overlap on the ROI. Fails if the ROI is outside all of them."""
    west, south, east, north = bbox
    best, best_area = None, 0.0
    for path in paths:
        with rasterio.open(path) as ds:
            b = ds.bounds
        overlap_w = max(0.0, min(east, b.right) - max(west, b.left))
        overlap_h = max(0.0, min(north, b.top) - max(south, b.bottom))
        area = overlap_w * overlap_h
        if area > best_area:
            best, best_area = path, area
    if best is None:
        raise ValueError(f"ROI {bbox} is outside every mosaic in {paths[0].parent}")
    return best


def valid_count_range(band_tags):
    """Inclusive DN range that means 'inundated vegetation in N observations'."""
    found = None
    for key, value in band_tags.items():
        match = re.fullmatch(r"DN_(\d+)-(\d+)", key)
        if match and "inundated" in value.lower():
            found = (int(match.group(1)), int(match.group(2)))
    if found is None:
        raise ValueError(f"No inundation DN range in band tags: {band_tags}")
    return found


def n_observations(band_tags, count_hi):
    text = " ".join(band_tags.values())
    match = re.search(r"(\d+) observations", text)
    return int(match.group(1)) if match else count_hi


def read_roi(path, bbox):
    west, south, east, north = bbox
    with rasterio.open(path) as ds:
        if ds.crs is None or ds.crs.to_epsg() != 4326:
            raise ValueError(f"{path.name} CRS is {ds.crs}; expected EPSG:4326")
        window = from_bounds(west, south, east, north, transform=ds.transform)
        window = window.round_offsets().round_lengths()
        window = window.intersection(Window(0, 0, ds.width, ds.height))
        if window.width <= 0 or window.height <= 0:
            raise ValueError(f"ROI {bbox} does not intersect {path.name}")
        data = ds.read(1, window=window)
        transform = ds.window_transform(window)
        tags = ds.tags(1)
    return data, transform, tags


def pixel_area_km2(transform, height):
    """WGS84 pixel area (km²) for each row. Degrees are not treated as metres."""
    rows = np.arange(height) + 0.5
    lat = transform.f + rows * transform.e
    lat_r = np.deg2rad(lat)
    m_lat = 111132.92 - 559.82 * np.cos(2 * lat_r) + 1.175 * np.cos(4 * lat_r)
    m_lon = 111412.84 * np.cos(lat_r) - 93.5 * np.cos(3 * lat_r)
    return (m_lon * abs(transform.a)) * (m_lat * abs(transform.e)) / 1e6


def slug(name):
    ascii_name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", ascii_name.lower()).strip("_")


def river_names(legend_csv=GLWD_LEGEND):
    with Path(legend_csv).open(newline="", encoding="utf-8") as handle:
        names = {int(row["GLWD_ID"]): row["Class_name"] for row in csv.DictReader(handle)}
    missing = [code for code in WATER_CLASS_IDS if code not in names]
    if missing:
        raise KeyError(f"GLWD legend has no class id {missing}")
    return names


def read_rivers(bbox, glwd_tif=GLWD_TIF):
    """GLWD class window for the ROI. Open-water pixels are opaque blue.

    Returns the RGBA image, its lon/lat extent, and the class ids that occur.
    """
    west, south, east, north = bbox
    with rasterio.open(glwd_tif) as ds:
        if ds.crs is None or ds.crs.to_epsg() != 4326:
            raise ValueError(f"{glwd_tif.name} CRS is {ds.crs}; expected EPSG:4326")
        window = from_bounds(west, south, east, north, transform=ds.transform)
        window = window.round_offsets().round_lengths()
        window = window.intersection(Window(0, 0, ds.width, ds.height))
        if window.width <= 0 or window.height <= 0:
            raise ValueError(f"ROI {bbox} does not intersect {glwd_tif.name}")
        codes = ds.read(1, window=window)
        transform = ds.window_transform(window)
    rgba = np.zeros((*codes.shape, 4), dtype=np.float32)
    present = []
    for code in WATER_CLASS_IDS:
        hit = codes == code
        if not hit.any():
            continue
        present.append(code)
        rgba[hit, :3] = np.array(WATER_RGB[code]) / 255.0
        rgba[hit, 3] = WATER_ALPHA
    height, width = codes.shape
    left = transform.c
    right = left + width * transform.a
    top = transform.f
    bottom = top + height * transform.e
    return rgba, (left, right, bottom, top), present


def alos2_flooded(roi_name, min_count=MIN_COUNT, out_dir=OUT_DIR):
    """Clip the ALOS-2 inundation mosaic to one ROI and save a figure.

    Returns a dict with the flooded area (km²) and the figure path.
    """
    name, bbox = load_roi(roi_name)
    path = pick_mosaic(bbox, mosaic_paths())
    data, transform, tags = read_roi(path, bbox)
    count_lo, count_hi = valid_count_range(tags)
    n_obs = n_observations(tags, count_hi)
    threshold = count_lo if min_count is None else max(int(min_count), count_lo)

    flooded = (data >= threshold) & (data <= count_hi)
    row_km2 = pixel_area_km2(transform, data.shape[0])
    flooded_km2 = float((flooded * row_km2[:, None]).sum())
    roi_km2 = float(row_km2.sum() * data.shape[1])

    freq = np.where(flooded, 100.0 * data / n_obs, np.nan)
    height, width = data.shape
    left = transform.c
    right = left + width * transform.a
    top = transform.f
    bottom = top + height * transform.e

    rivers, river_extent, water_present = read_rivers(bbox)
    names = river_names()
    cmap = plt.cm.inferno.copy()
    cmap.set_bad((0, 0, 0, 0))
    west, south, east, north = bbox
    fig_w = 7.0
    fig_h = float(np.clip(fig_w * (north - south) / (east - west), 4.5, 9.5))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.patch.set_facecolor("black")
    ax.set_facecolor("black")
    ax.imshow(
        rivers,
        extent=river_extent,
        origin="upper",
        interpolation="nearest",
        zorder=1,
    )
    image = ax.imshow(
        freq,
        extent=(left, right, bottom, top),
        origin="upper",
        cmap=cmap,
        vmin=0,
        vmax=100,
        interpolation="nearest",
        zorder=2,
    )
    ax.set_xlim(west, east)
    ax.set_ylim(south, north)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(
        f"{name}\n"
        f"Inundated vegetation in ≥ {threshold}/{n_obs} observations: "
        f"{flooded_km2:,.0f} km²"
    )
    ax.tick_params(colors="white")
    ax.xaxis.label.set_color("white")
    ax.yaxis.label.set_color("white")
    ax.title.set_color("white")
    for spine in ax.spines.values():
        spine.set_color("white")
    if water_present:
        legend = ax.legend(
            handles=[
                Patch(facecolor=np.array(WATER_RGB[code]) / 255.0, edgecolor="none", label=names[code])
                for code in water_present
            ],
            loc="upper center",
            bbox_to_anchor=(0.5, -0.14),
            ncol=2,
            fontsize=7,
        )
        legend.get_frame().set_facecolor("black")
        legend.get_frame().set_edgecolor("white")
        for text in legend.get_texts():
            text.set_color("white")
    colorbar = fig.colorbar(image, ax=ax, fraction=0.04, pad=0.02)
    colorbar.set_label(f"% of {n_obs} observations (2015–2025)", color="white")
    colorbar.ax.yaxis.set_tick_params(color="white")
    plt.setp(colorbar.ax.yaxis.get_ticklabels(), color="white")
    fig.tight_layout()

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    figure_path = out_dir / f"{slug(name)}_alos2_inundation.png"
    fig.savefig(figure_path, dpi=150, facecolor=fig.get_facecolor(), bbox_inches="tight")
    plt.show()

    print(
        f"{name}: {flooded_km2:,.1f} km² inundated "
        f"({100 * flooded_km2 / roi_km2:.1f}% of {roi_km2:,.0f} km² ROI), "
        f"threshold ≥ {threshold}/{n_obs}, mosaic {path.name}"
    )
    print(f"figure: {figure_path.resolve()}")
    return {
        "name": name,
        "flooded_km2": flooded_km2,
        "roi_km2": roi_km2,
        "threshold": threshold,
        "n_obs": n_obs,
        "mosaic": str(path),
        "figure": str(figure_path),
    }

In [ ]:
results = [alos2_flooded(name) for name in ROI_NAMES]

# ALOS-2 inundation on Google Terrain

Same ROI list and the same inundation layer as above. The GLWD classes are left off. The basemap is the Google terrain tiles (`lyrs=p`), warped onto the ALOS grid. Pixels with no inundated vegetation stay transparent so the terrain shows through.

In [ ]:
import math
import urllib.request

from PIL import Image
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.warp import reproject

# Google terrain (hillshade, roads, labels). Tiles are cached under outputs/.
TERRAIN_URL = "https://mt1.google.com/vt/lyrs=p&hl=en&x={x}&y={y}&z={z}"
TERRAIN_CACHE = OUT_DIR / "terrain_cache"
TERRAIN_ZOOM = None  # None picks a zoom that stays under TERRAIN_MAX_TILES
TERRAIN_MAX_TILES = 48


def terrain_zoom(west, south, east, north, max_tiles=TERRAIN_MAX_TILES):
    """Finest zoom whose tile count for the ROI is at most max_tiles."""
    chosen = 8
    for zoom in range(14, 7, -1):
        n = 2**zoom

        def tile_xy(lon, lat, n=n):
            x = (lon + 180.0) / 360.0 * n
            lat_r = math.radians(min(max(lat, -85.0), 85.0))
            y = (1.0 - math.asinh(math.tan(lat_r)) / math.pi) / 2.0 * n
            return x, y

        x0, y_north = tile_xy(west, north)
        x1, y_south = tile_xy(east, south)
        tiles = (int(math.floor(x1)) - int(math.floor(x0)) + 1) * (
            int(math.floor(y_south)) - int(math.floor(y_north)) + 1
        )
        if tiles <= max_tiles:
            return zoom
        chosen = zoom
    return chosen


def terrain_mosaic(west, south, east, north, zoom):
    """RGB mosaic in EPSG:3857 covering the ROI, and its affine transform."""
    n = 2**zoom

    def tile_xy(lon, lat):
        x = (lon + 180.0) / 360.0 * n
        lat_r = math.radians(min(max(lat, -85.0), 85.0))
        y = (1.0 - math.asinh(math.tan(lat_r)) / math.pi) / 2.0 * n
        return x, y

    x0, y_north = tile_xy(west, north)
    x1, y_south = tile_xy(east, south)
    c0, c1 = max(int(math.floor(x0)), 0), min(int(math.floor(x1)), n - 1)
    r0, r1 = max(int(math.floor(y_north)), 0), min(int(math.floor(y_south)), n - 1)
    height = (r1 - r0 + 1) * 256
    width = (c1 - c0 + 1) * 256
    mosaic = np.full((height, width, 3), 230, dtype=np.uint8)
    TERRAIN_CACHE.mkdir(parents=True, exist_ok=True)
    for row in range(r0, r1 + 1):
        for col in range(c0, c1 + 1):
            cache = TERRAIN_CACHE / f"{zoom}_{col}_{row}.png"
            if not cache.exists():
                url = TERRAIN_URL.format(x=col, y=row, z=zoom)
                request = urllib.request.Request(url, headers={"User-Agent": "biomass-hackathon-map/1.0"})
                cache.write_bytes(urllib.request.urlopen(request, timeout=60).read())
            rgb = np.asarray(Image.open(cache).convert("RGB"))
            i0, j0 = (row - r0) * 256, (col - c0) * 256
            mosaic[i0 : i0 + 256, j0 : j0 + 256] = rgb

    def corner(col, row):
        lon = col / n * 360.0 - 180.0
        lat = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * row / n))))
        half = 20037508.342789244
        x = lon / 180.0 * half
        y = math.log(math.tan(math.pi / 4.0 + math.radians(lat) / 2.0)) * half / math.pi
        return x, y

    left, top = corner(c0, r0)
    right, bottom = corner(c1 + 1, r1 + 1)
    transform = transform_from_bounds(left, bottom, right, top, width, height)
    return mosaic, transform


def warp_terrain(mosaic, src_transform, dst_shape, dst_transform):
    """Bilinear sample of the EPSG:3857 mosaic onto the ALOS EPSG:4326 grid."""
    warped = np.zeros((*dst_shape, 3), dtype=np.uint8)
    for band in range(3):
        reproject(
            source=mosaic[:, :, band],
            destination=warped[:, :, band],
            src_transform=src_transform,
            src_crs=CRS.from_epsg(3857),
            dst_transform=dst_transform,
            dst_crs=CRS.from_epsg(4326),
            resampling=Resampling.bilinear,
        )
    return warped


def alos2_on_terrain(roi_name, min_count=MIN_COUNT, out_dir=OUT_DIR, zoom=TERRAIN_ZOOM):
    """ALOS-2 inundation over a Google terrain basemap. One figure per ROI."""
    name, bbox = load_roi(roi_name)
    west, south, east, north = bbox
    path = pick_mosaic(bbox, mosaic_paths())
    data, transform, tags = read_roi(path, bbox)
    count_lo, count_hi = valid_count_range(tags)
    n_obs = n_observations(tags, count_hi)
    threshold = count_lo if min_count is None else max(int(min_count), count_lo)
    flooded = (data >= threshold) & (data <= count_hi)
    freq = np.where(flooded, 100.0 * data / n_obs, np.nan).astype(np.float32)

    zoom = terrain_zoom(west, south, east, north) if zoom is None else int(zoom)
    mosaic, terrain_transform = terrain_mosaic(west, south, east, north, zoom)
    basemap = warp_terrain(mosaic, terrain_transform, data.shape, transform)

    height, width = data.shape
    left = transform.c
    right = left + width * transform.a
    top = transform.f
    bottom = top + height * transform.e
    cmap = plt.cm.inferno.copy()
    cmap.set_bad((0, 0, 0, 0))

    fig_w = 7.0
    fig_h = float(np.clip(fig_w * (north - south) / (east - west), 4.5, 9.5))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.imshow(basemap, extent=(left, right, bottom, top), origin="upper", interpolation="nearest", zorder=1)
    image = ax.imshow(
        freq,
        extent=(left, right, bottom, top),
        origin="upper",
        cmap=cmap,
        vmin=0,
        vmax=100,
        interpolation="nearest",
        zorder=2,
    )
    ax.set_xlim(west, east)
    ax.set_ylim(south, north)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"{name}\nALOS-2 inundated vegetation on Google terrain")
    colorbar = fig.colorbar(image, ax=ax, fraction=0.04, pad=0.02)
    colorbar.set_label(f"% of {n_obs} observations (2015–2025)")
    ax.text(
        0.01,
        0.01,
        "Basemap: Google Terrain",
        transform=ax.transAxes,
        fontsize=7,
        color="0.15",
        ha="left",
        va="bottom",
    )
    fig.tight_layout()

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    figure_path = out_dir / f"{slug(name)}_alos2_terrain.png"
    fig.savefig(figure_path, dpi=150)
    plt.show()
    print(f"{name}: terrain zoom {zoom}, mosaic {path.name}")
    print(f"figure: {figure_path.resolve()}")
    return {"name": name, "zoom": zoom, "figure": str(figure_path)}


terrain_results = [alos2_on_terrain(name) for name in ROI_NAMES]
